# 1. Open datasets

In [1]:
#%pip install "setuptools<82"
#%pip install PyTDC

%pip install -r ../requirements.txt

  Using cached pytdc-1.1.15.tar.gz (154 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached rdkit-2025.9.3-cp312-cp312-macosx_10_15_x86_64.whl.metadata (4.2 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached accelerate-0.33.0-py3-none-any.whl.metadata (18 kB)
  Using cached dataclasses-0.6-py3-none-any.whl.metadata (3.0 kB)
  Using cached datasets-2.19.2-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.2-py3-none-any.whl.metadata (9.3 kB)
  Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl.metadata (4.9 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata (61 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached transformers-4.50.3-py3-none-any.

In [2]:
from pathlib import Path

from tdc.single_pred import ADME, Tox
import pandas as pd

# Save datasets in the project-level data/raw directory
raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Load regression dataset
solubility = ADME(name="Solubility_AqSolDB")
df_reg = solubility.get_data()
df_reg.to_csv(raw_dir / "solubility_aqsoldb.tab", sep="\t", index=False)

# Load classification dataset
herg = Tox(name="hERG")
df_clf = herg.get_data()
df_clf.to_csv(raw_dir / "herg.tab", sep="\t", index=False)

# Preview datasets and report shapes
display(df_reg.head(10))
display(df_clf.head(10))
print("Regression:", df_reg.shape)
print("Classification:", df_clf.shape)

ModuleNotFoundError: No module named 'tdc'

# 2. Check for missing or invalid values

In [ ]:
import numpy as np, pandas as pd

In [ ]:
df_reg = solubility.get_data()

df_reg = df_reg.rename(columns={
    "Drug": "SMILES",
    "Y": "Solubility"
})

print("Dataset: Solubility_AqSolDB")
print("Target: Aqueous solubility")
print("Original rows:", len(df_reg))


print("Missing SMILES:", df_reg["SMILES"].isna().sum())
print("Missing solubility:", df_reg["Solubility"].isna().sum())

df_clf = herg.get_data()

df_clf = df_clf.rename(columns={
    "Drug": "SMILES",
    "Y": "HerG"
})

print("Dataset: HerG")
print("Target: Toxicity")
print("Original rows:", len(df_clf))


print("Missing SMILES:", df_clf["SMILES"].isna().sum())
print("Missing HerG:", df_clf["HerG"].isna().sum())

Dataset: Solubility_AqSolDB
Target: Aqueous solubility
Original rows: 9982
Missing SMILES: 0
Missing solubility: 0
Dataset: HerG
Target: Toxicity
Original rows: 655
Missing SMILES: 0
Missing HerG: 0


In [ ]:
initial_rows = len(df_reg)

df_reg = df_reg.dropna(subset=["SMILES", "Solubility"]).copy()

print("Rows removed:", initial_rows - len(df_reg))
print("Rows remaining:", len(df_reg))
display (df_reg.head())

initial_rows = len(df_clf)

df_clf = df_clf.dropna(subset=["SMILES", "HerG"]).copy()

print("Rows removed:", initial_rows - len(df_clf))
print("Rows remaining:", len(df_clf))
display (df_clf.head())

Rows removed: 0
Rows remaining: 9982


,Drug_ID,SMILES,Solubility
0,"N,N,N-trimethyloctadecan-1-aminium bromide",CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-],-3.616127
1,Benzo[cd]indol-2(1H)-one,O=C1Nc2cccc3cccc1c23,-3.254767
2,4-chlorobenzaldehyde,O=Cc1ccc(Cl)cc1,-2.177078
3,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...,-3.924409
4,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...,-4.662065


Rows removed: 0
Rows remaining: 655


,Drug_ID,SMILES,HerG
0,DEMETHYLASTEMIZOLE,Oc1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccc(F)cc3)CC2)cc1,1.0
1,GBR-12909,Fc1ccc(C(OCC[NH+]2CC[NH+](CCCc3ccccc3)CC2)c2cc...,1.0
2,LY-97241,CCCCCCCN(CC)CCCCc1ccc([N+](=O)[O-])cc1,1.0
3,CLOFILIUM PHOSPHATE,CCCCCCC[N+](CC)(CC)CCCCc1ccc(Cl)cc1.CCCCCCC[N+...,1.0
4,FLUSPIRILENE,O=C1NCN(c2ccccc2)C12CC[NH+](CCCC(c1ccc(F)cc1)c...,1.0


# 3. Check for the SMILES 

In [3]:
from rdkit import Chem

# Set the column names
smiles_col = 'smiles'
target_col = 'HerG'  
# Check the target values
print("Target distribution:")
print(df_clf[target_col].value_counts(dropna=False))

# Count missing targets and SMILES
print("Missing targets:", df_clf[target_col].isna().sum())
print("Missing SMILES:", df_clf[smiles_col].isna().sum())

# Remove rows with missing targets
df_clf = df_clf.dropna(subset=[target_col]).copy()

print("Rows after removing missing targets:", len(df_clf))

ModuleNotFoundError: No module named 'rdkit'

# 4. Create a molecule column and check invalid SMILES

In [ ]:
# Convert SMILES into molecule objects
df_clf['Molecule'] = df_clf[smiles_col].apply(
    lambda x: Chem.MolFromSmiles(x) if isinstance(x, str) else None
)

# Count invalid SMILES
invalid_smiles = df_clf['Molecule'].isna().sum()
print("Invalid SMILES:", invalid_smiles)

# Remove invalid SMILES
df_clf = df_clf.dropna(subset=['Molecule']).copy()

print("Rows after removing invalid SMILES:", len(df_clf))

# 5. Generate canonical SMILES

In [ ]:
# Function to canonicalize SMILES
def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        canonical_smiles = Chem.MolToSmiles(mol, canonical=True)
        return canonical_smiles
    else:
        return None

# Apply the function to the dataset
df_clf['canonical_smiles'] = df_clf[smiles_col].apply(canonicalize_smiles)

df_clf.head()

# 6. Identify duplicates and conflicting targets

In [ ]:
# Count duplicate structures
print("Duplicate structures:", df_clf['canonical_smiles'].duplicated().sum())

# Find structures with different target values
target_counts = df_clf.groupby('canonical_smiles')[target_col].nunique()

conflicting_smiles = target_counts[target_counts > 1].index

print("Structures with conflicting targets:", len(conflicting_smiles))

# 7. Plot the target distribution

In [4]:
import matplotlib.pyplot as plt

# Plot the target distribution
df_clf[target_col].value_counts().sort_index().plot(kind='bar')

plt.title('hERG Target Distribution')
plt.xlabel('Target label')
plt.ylabel('Number of molecules')
plt.xticks(rotation=0)
plt.show()

# Check class proportions
print("Class proportions:")
print(df_clf[target_col].value_counts(normalize=True))

NameError: name 'df_clf' is not defined